In [2]:
# CS 340 Project Two Dashboard
# Jesse Carles
# Grazioso Salvare Rescue Animal Dashboard

# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
from jupyter_dash import JupyterDash

import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output

import plotly.express as px
import pandas as pd
import base64

# Import the reusable CRUD module from Project One
from CRUD_Python_Module import AnimalShelter


# ------------------------------------------------------------
# Configure JupyterDash for Codio
# ------------------------------------------------------------
JupyterDash.infer_jupyter_proxy_config()


# ------------------------------------------------------------
# MongoDB / Model
# ------------------------------------------------------------
USERNAME = "aacuser"
PASSWORD = "YOUR_PASSWORD_HERE"

db = AnimalShelter(USERNAME, PASSWORD)


# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------
def get_dataframe(query):
    """
    Retrieve records from MongoDB using the reusable CRUD module
    and convert the results to a Pandas DataFrame.
    """

    records = db.read(query)
    dataframe = pd.DataFrame.from_records(records)

    # MongoDB ObjectId values cannot be serialized by Dash.
    if "_id" in dataframe.columns:
        dataframe.drop(columns=["_id"], inplace=True)

    return dataframe


# Retrieve all records for initial/reset state
df = get_dataframe({})


# ------------------------------------------------------------
# Grazioso Salvare logo
# ------------------------------------------------------------
image_filename = "Grazioso Salvare Logo.png"

with open(image_filename, "rb") as image_file:
    encoded_image = base64.b64encode(image_file.read()).decode()


# ------------------------------------------------------------
# Build Dashboard
# ------------------------------------------------------------
app = JupyterDash(__name__)


app.layout = html.Div([

    # --------------------------------------------------------
    # Branding / Unique Identifier
    # --------------------------------------------------------
    html.Div(
        [
            html.Img(
                src="data:image/png;base64,{}".format(encoded_image),
                style={
                    "height": "150px",
                    "display": "block",
                    "margin": "0 auto"
                }
            ),

            html.H1(
                "Grazioso Salvare Rescue Animal Dashboard",
                style={"textAlign": "center"}
            ),

            html.H3(
                "Jesse Carles - CS 340",
                style={"textAlign": "center"}
            )
        ]
    ),

    html.Hr(),

    # --------------------------------------------------------
    # Interactive Filtering Controls
    # --------------------------------------------------------
    html.H3("Rescue Type Filter"),

    dcc.RadioItems(
        id="filter-type",

        options=[
            {
                "label": "Water Rescue",
                "value": "water"
            },
            {
                "label": "Mountain or Wilderness Rescue",
                "value": "mountain"
            },
            {
                "label": "Disaster or Individual Tracking",
                "value": "disaster"
            },
            {
                "label": "Reset",
                "value": "reset"
            }
        ],

        value="reset",

        labelStyle={
            "display": "inline-block",
            "marginRight": "25px"
        }
    ),

    html.Hr(),

    # --------------------------------------------------------
    # Interactive Data Table
    # --------------------------------------------------------
    dash_table.DataTable(

        id="datatable-id",

        columns=[
            {
                "name": column,
                "id": column,
                "deletable": False,
                "selectable": True
            }
            for column in df.columns
        ],

        data=df.to_dict("records"),

        row_selectable="single",

        # Select first row by default so map always has a record
        selected_rows=[0],

        page_action="native",
        page_current=0,
        page_size=10,

        sort_action="native",
        sort_mode="multi",

        filter_action="native",

        style_table={
            "overflowX": "auto"
        },

        style_cell={
            "textAlign": "left",
            "padding": "6px",
            "minWidth": "100px",
            "width": "125px",
            "maxWidth": "220px",
            "whiteSpace": "normal",
            "height": "auto"
        },

        style_header={
            "fontWeight": "bold"
        }
    ),

    html.Br(),

    html.Hr(),

    # --------------------------------------------------------
    # Charts
    # --------------------------------------------------------
    html.Div(
        className="row",

        style={
            "display": "flex",
            "flexWrap": "wrap"
        },

        children=[

            # Pie chart
            html.Div(
                id="graph-id",

                style={
                    "width": "50%",
                    "minWidth": "500px"
                }
            ),

            # Map
            html.Div(
                id="map-id",

                style={
                    "width": "50%",
                    "minWidth": "500px"
                }
            )
        ]
    )
])


# ============================================================
# CONTROLLER
# ============================================================


# ------------------------------------------------------------
# Filter DataTable using MongoDB queries
# ------------------------------------------------------------
@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value")
)
def update_dashboard(filter_type):

    # WATER RESCUE
    if filter_type == "water":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },

            "sex_upon_outcome": "Intact Female",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # MOUNTAIN / WILDERNESS RESCUE
    elif filter_type == "mountain":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # DISASTER / INDIVIDUAL TRACKING
    elif filter_type == "disaster":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    # RESET
    else:

        query = {}

    filtered_df = get_dataframe(query)

    return filtered_df.to_dict("records")


# ------------------------------------------------------------
# Dynamic Pie Chart
# ------------------------------------------------------------
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_graphs(view_data):

    if view_data is None or len(view_data) == 0:

        return html.Div(
            "No data available for chart.",
            style={
                "textAlign": "center",
                "padding": "40px"
            }
        )

    dff = pd.DataFrame(view_data)

    if "breed" not in dff.columns:

        return html.Div("Breed information unavailable.")

    # Count breeds represented in current filtered table
    breed_counts = (
        dff["breed"]
        .fillna("Unknown")
        .value_counts()
        .reset_index()
    )

    breed_counts.columns = [
        "breed",
        "count"
    ]

    figure = px.pie(
        breed_counts,
        values="count",
        names="breed",
        title="Breed Distribution"
    )

    figure.update_layout(
        legend_title_text="Breed"
    )

    return [
        dcc.Graph(
            figure=figure
        )
    ]


# ------------------------------------------------------------
# Highlight selected columns
# ------------------------------------------------------------
@app.callback(
    Output(
        "datatable-id",
        "style_data_conditional"
    ),

    Input(
        "datatable-id",
        "selected_columns"
    )
)
def update_styles(selected_columns):

    if selected_columns is None:
        selected_columns = []

    return [

        {
            "if": {
                "column_id": column
            },

            "backgroundColor": "#D2F3FF"
        }

        for column in selected_columns
    ]


# ------------------------------------------------------------
# Dynamic Geolocation Map
# ------------------------------------------------------------
@app.callback(
    Output(
        "map-id",
        "children"
    ),

    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),

        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]
)
def update_map(view_data, selected_rows):

    if view_data is None or len(view_data) == 0:

        return html.Div(
            "No location data available.",
            style={
                "textAlign": "center",
                "padding": "40px"
            }
        )

    dff = pd.DataFrame(view_data)

    # Default to the first row
    if selected_rows is None or len(selected_rows) == 0:
        row = 0
    else:
        row = selected_rows[0]

    # Prevent an invalid row selection after filtering
    if row >= len(dff):
        row = 0

    try:

        latitude = float(
            dff.iloc[row]["location_lat"]
        )

        longitude = float(
            dff.iloc[row]["location_long"]
        )

    except (
        KeyError,
        TypeError,
        ValueError
    ):

        return html.Div(
            "Selected animal does not contain valid location data."
        )

    animal_name = str(
        dff.iloc[row].get(
            "name",
            "Unknown"
        )
    )

    animal_breed = str(
        dff.iloc[row].get(
            "breed",
            "Unknown"
        )
    )

    animal_id = str(
        dff.iloc[row].get(
            "animal_id",
            "Unknown"
        )
    )

    return [

        dl.Map(

            style={
                "width": "100%",
                "height": "500px"
            },

            center=[
                latitude,
                longitude
            ],

            zoom=10,

            children=[

                dl.TileLayer(
                    id="base-layer-id"
                ),

                dl.Marker(

                    position=[
                        latitude,
                        longitude
                    ],

                    children=[

                        dl.Tooltip(
                            animal_breed
                        ),

                        dl.Popup([

                            html.H4(
                                "Animal Information"
                            ),

                            html.P(
                                f"Name: {animal_name}"
                            ),

                            html.P(
                                f"Breed: {animal_breed}"
                            ),

                            html.P(
                                f"Animal ID: {animal_id}"
                            )
                        ])
                    ]
                )
            ]
        )
    ]


# ------------------------------------------------------------
# Run application in Codio
# ------------------------------------------------------------
app.run_server(mode="external")

Dash app running on https://kineticcake-analyzeedison-3000.codio.io/proxy/8050/
